# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature vector built by `features()` in wk.py: log1p on heavy-tailed counts, median fill, one-hot categoricals.

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
X = features(df); print(X.shape); X.head(2)

(30000, 30)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,sessions_90d,ai_sessions_90d,days_with_impressions,...,competition_level_MEDIUM,competition_level_unknown,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,2.397895,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,0,0,0,0,1,0,0,0,1,0
1,4.510860,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,0,0,0,0,1,0,1,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

All NUM features are snapshot properties or 90-day aggregates available at scoring time; categoricals are page metadata. Missing numerics → median fill; missing categoricals → 'unknown'.

In [2]:
print(df[NUM].isna().mean().round(3).sort_values(ascending=False).head(6))

char_count       0.257
word_count       0.257
competition      0.082
search_volume    0.082
cpc              0.082
scroll_rate      0.004
dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Leakage test:** train the same RF on a client-holdout split with (a) my safe features, (b) safe + trend_pct, (c) safe + 30-day trend windows. A big jump means leakage.

In [3]:
tr, te = client_split(df)
y = df.y.values
def run(X):
    m = models()['random_forest'].fit(X.iloc[tr], y[tr]); return evaluate(y[te], m.predict_proba(X.iloc[te])[:,1])
Xs = features(df)
print('safe          ', run(Xs))
print('+trend_pct    ', run(Xs.assign(trend_pct=df.trend_pct.fillna(0))))
Xw = Xs.copy()
for c in ['impressions_last_30d','impressions_prev_30d','clicks_last_30d','clicks_prev_30d']: Xw[c]=np.log1p(df[c])
print('+30d windows  ', run(Xw))

safe           {'roc_auc': 0.626, 'avg_precision': 0.617, 'p@50': 0.74, 'p@200': 0.68, 'base_rate': 0.511, 'n': 6163}


+trend_pct     {'roc_auc': 1.0, 'avg_precision': 1.0, 'p@50': 1.0, 'p@200': 1.0, 'base_rate': 0.511, 'n': 6163}


+30d windows   {'roc_auc': 0.895, 'avg_precision': 0.903, 'p@50': 1.0, 'p@200': 1.0, 'base_rate': 0.511, 'n': 6163}


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Refused: everything in EXCLUDED (printed). trend_pct / trend_direction define the label; the 30-day windows are the label's inputs; IDs are not page properties.

In [4]:
for k,v in EXCLUDED.items(): print(k,'-',v)

content_id - pseudonymous ID - grouping only
client_id - pseudonymous ID - grouping only
trend_direction - label source
trend_pct - label-derived
impressions_last_30d - trend input window
clicks_last_30d - trend input window
sessions_last_30d - trend input window
impressions_prev_30d - trend input window
clicks_prev_30d - trend input window
sessions_prev_30d - trend input window
provider_used - generation metadata, mostly blank
model_used - generation metadata, not a page property


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.